# mc_single_arm 输出数据可视化

该 notebook 用于可视化 `mc_single_arm.py` 生成的 `worksim/*.npz` 数据（数组名：`ntuple`）。

默认展示：
- 焦平面位置散点图 (`x_fp` vs `y_fp`)
- 初始/重建 `dpp` 分布对比直方图
- 初始角度二维密度图 (`dth_init` vs `dph_init`)
- `stop_id` 计数图


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# 在这里修改为你的输出文件名（不带 .npz 后缀）
output_stem = "hms_40deg_carbon_1560"

repo_root = Path.cwd().resolve().parent
npz_path = repo_root / "worksim" / f"{output_stem}.npz"
if not npz_path.exists():
    raise FileNotFoundError(
        f"未找到文件: {npz_path}\n"
        "请先运行: cd python && python mc_single_arm.py <input_stem>"
    )

data = np.load(npz_path)
if "ntuple" not in data:
    raise KeyError(f"{npz_path} 中未找到数组 'ntuple'")

ntuple = data["ntuple"]
print(f"loaded: {npz_path}")
print(f"shape: {ntuple.shape}")
if ntuple.ndim != 2 or ntuple.shape[1] < 21:
    raise ValueError("ntuple 格式异常，期望二维数组且至少包含 21 列")


In [ ]:
# 列定义（详见 python/README.md 的 Ntuple format）
COL = {
    "x_fp": 0,
    "y_fp": 1,
    "dph_init": 6,
    "dth_init": 7,
    "dpp_init": 9,
    "dpp_recon": 14,
    "stop_id": 20,
}

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# 1) 焦平面位置
axes[0, 0].scatter(ntuple[:, COL["x_fp"]], ntuple[:, COL["y_fp"]], s=2, alpha=0.3)
axes[0, 0].set_title("Focal Plane: x_fp vs y_fp")
axes[0, 0].set_xlabel("x_fp")
axes[0, 0].set_ylabel("y_fp")

# 2) dpp 初始/重建对比
axes[0, 1].hist(ntuple[:, COL["dpp_init"]], bins=80, alpha=0.55, label="dpp_init")
axes[0, 1].hist(ntuple[:, COL["dpp_recon"]], bins=80, alpha=0.55, label="dpp_recon")
axes[0, 1].set_title("dpp Distribution")
axes[0, 1].set_xlabel("dpp (%)")
axes[0, 1].set_ylabel("counts")
axes[0, 1].legend()

# 3) 初始角度二维密度
h = axes[1, 0].hexbin(
    ntuple[:, COL["dth_init"]],
    ntuple[:, COL["dph_init"]],
    gridsize=70,
    mincnt=1,
    cmap="viridis",
)
axes[1, 0].set_title("Initial Angles: dth_init vs dph_init")
axes[1, 0].set_xlabel("dth_init (rad)")
axes[1, 0].set_ylabel("dph_init (rad)")
fig.colorbar(h, ax=axes[1, 0], label="counts")

# 4) stop_id 统计
stop_ids = ntuple[:, COL["stop_id"]].astype(int)
uniq, counts = np.unique(stop_ids, return_counts=True)
axes[1, 1].bar(uniq, counts, width=0.9)
axes[1, 1].set_title("stop_id Counts")
axes[1, 1].set_xlabel("stop_id")
axes[1, 1].set_ylabel("counts")

plt.tight_layout()
plt.show()
